# Does the answer depend on my choices?

*Question, Intuition, Math, Code, Assumptions, How it breaks*

Four numbers in a config file decide who the best footballer of the last
twenty-five years is.

```
gate_percentile = 40      the floor every requirement must clear
min_minutes     = 900     how much football a season needs to count
min_seasons     = 3       how many seasons a career needs to be ranked
prior_minutes   = 900     how hard a small sample gets pulled to the mean
```

I picked all four. Not one of them is derived from anything, and until now
nothing in this project defended any of them. That is the most obvious hole in a
book that spends thirteen chapters demanding evidence for everything else.

So this chapter attacks them.

## 1. Question

If I had picked different constants, would I have got a different answer?

## 2. Intuition

There is a specific failure this is looking for. A result that moves when you
jiggle a threshold is not a result about football, it is a result about the
threshold. The gate is at the 40th percentile; if the winner changes at the 45th,
then "Messi is the best footballer" really means "Messi is the best footballer
when you draw the line exactly here", and nobody would accept that phrased
honestly.

The test is mechanical. Re-run the whole thing at every plausible setting and see
what survives. There is no cleverness in it, which is why there is no excuse for
not having done it.

## 3. Math

Two things get measured for each setting, and they answer different questions.

**Who wins**, and how much of the top ten survives. This is the question a reader
cares about.

**Spearman's $\rho$ against the shipped ranking**, computed on the players
present in both. Ranks rather than scores, because the scores are re-standardised
each run and are not comparable across settings.

That second measure has a trap in it worth naming. It is computed on the
*intersection*, so a setting that throws half the population out can still score
$\rho \approx 1$ on whoever is left. It says "the survivors are ordered the same
way", not "nothing changed". The qualifier counts are printed next to it for
exactly that reason.

## 4. Code

Everything below re-derives the ranking from the committed sample, using the
same `_rank_group` the pipeline calls. First, proof that this reproduces what the
book actually publishes, because a sensitivity analysis run on different
machinery would be worthless.

In [1]:
import dataclasses

import pandas as pd

from gambeta import kit, needs
from gambeta.cli import _rank_group

cfg0 = kit.load()
seasons = pd.read_parquet("../data/sample/player_season_scored.parquet")
published = pd.read_parquet("../data/sample/ranking.parquet")


def rank_with(cfg: kit.Config) -> pd.DataFrame:
    """Re-derive the outfield ranking under one configuration."""
    return _rank_group(seasons[seasons["minutes"] >= cfg.min_minutes], needs.OUTFIELD, cfg)[0]


baseline = rank_with(cfg0)
mine = list(baseline[baseline["qualified"]]["player"])
theirs = list(published[published["qualified"]]["player"])
same = mine == theirs
print(f"reproduces the published ranking exactly: {same}")
print(f"{len(baseline):,} ranked, {int(baseline['qualified'].sum())} qualified")

reproduces the published ranking exactly: True
5,508 ranked, 342 qualified


Now the sweep. Every setting I would defend in an argument, and a few I would
not.

In [2]:
def describe(parameter: str, value: object, ranking: pd.DataFrame) -> dict:
    q, qb = ranking[ranking["qualified"]], baseline[baseline["qualified"]]
    kept = len(set(q.head(10)["player"]) & set(qb.head(10)["player"]))
    a = q.reset_index(drop=True).reset_index().set_index("player")["index"]
    b = qb.reset_index(drop=True).reset_index().set_index("player")["index"]
    both = a.index.intersection(b.index)
    # Spearman without scipy is Pearson on the ranks.
    rho = a[both].rank().corr(b[both].rank()) if len(both) > 2 else float("nan")
    return {
        "parameter": parameter,
        "value": value,
        "ranked": len(ranking),
        "qualified": len(q),
        "winner": q.iloc[0]["player"],
        "top 10 kept": f"{kept}/10",
        "rho": round(rho, 4),
    }


SWEEP = {
    "gate_percentile": (20, 30, 40, 50, 60),
    "min_seasons": (2, 3, 5, 8),
    "min_minutes": (900, 1350, 1800),
    "prior_minutes": (0.0, 900.0, 3600.0),
}

rows = [describe("as shipped", "", baseline)]
for parameter, values in SWEEP.items():
    for value in values:
        tuned = dataclasses.replace(cfg0, **{parameter: value})
        rows.append(describe(parameter, value, rank_with(tuned)))

pd.DataFrame(rows).style.format({"rho": "{:.4f}"}).hide(axis="index")

parameter,value,ranked,qualified,winner,top 10 kept,rho
as shipped,,5508,342,Lionel Messi,10/10,1.0000
gate_percentile,20,5508,1829,Lionel Messi,10/10,0.9875
gate_percentile,30,5508,845,Lionel Messi,10/10,0.9998
gate_percentile,40,5508,342,Lionel Messi,10/10,1.0000
gate_percentile,50,5508,136,Lionel Messi,7/10,1.0000
gate_percentile,60,5508,38,Lionel Messi,7/10,1.0000
min_seasons,2,7218,512,Lionel Messi,10/10,0.9996
min_seasons,3,5508,342,Lionel Messi,10/10,1.0000
min_seasons,5,3356,229,Lionel Messi,10/10,0.9991
min_seasons,8,1510,87,Lionel Messi,7/10,0.9964


Read the `winner` column first. **Lionel Messi is first under every single
setting**, including the ones I would argue against: a gate so tight that
thirty-eight players survive, and a career floor so high that most of the modern
game is excluded.

That is the result this chapter exists to produce. The constants decide **how
many** players qualify, which swings from 1,829 to 38. They do not decide who
wins.

### Where it does move

The top ten holds completely across the settings I would defend, and loses three
places at the extremes. It is worth knowing which three, because they are the
same three every time.

In [3]:
top10 = list(baseline[baseline["qualified"]].head(10)["player"])

for parameter, value in [("gate_percentile", 50), ("min_seasons", 8), ("min_minutes", 1800)]:
    r = rank_with(dataclasses.replace(cfg0, **{parameter: value}))
    gone = [p for p in top10 if p not in list(r[r["qualified"]].head(10)["player"])]
    print(f"{parameter} = {value}")
    for player in gone:
        row = r[r["player"] == player]
        reason = row["failed"].item() if len(row) else "not ranked at all"
        print(f"    {player:20s} {reason}")
    print()

gate_percentile = 50
    Kylian Mbappé        reliability
    Robert Lewandowski   discipline
    Karim Benzema        reliability



min_seasons = 8
    Kylian Mbappé        reliability
    Erling Haaland       not ranked at all
    Karim Benzema        reliability



min_minutes = 1800
    Kylian Mbappé        availability, reliability
    Robert Lewandowski   discipline
    Karim Benzema        reliability



Mbappé and Benzema, every time, on **reliability**. Lewandowski on discipline.
Haaland simply has not played enough seasons yet to survive a career floor of
eight.

That is not really a finding about the thresholds. It is a finding about
`reliability`, which is starts divided by appearances, and which this project
already has an open question about. Tighten any threshold and the same
requirement is what gives way first. A sensitivity sweep is supposed to tell you
which of your choices is load-bearing, and it did: the answer is none of the four
constants, it is a requirement I have not fixed yet.

## 5. Assumptions

1. **The published sample is already filtered at 900 minutes.** So the minutes
   floor can be raised here but not lowered, and the 450-minute case is missing
   from the table above. Run it against `vault/clean` if you want that half.
2. **Each setting re-standardises.** Scores are not comparable between rows,
   which is why everything above compares ranks.
3. **One parameter moves at a time.** Interactions are not tested. A tighter gate
   combined with a higher minutes floor could behave differently than either
   alone.

## 6. How it breaks

One of those four constants does nothing at all, and this sweep is how I found
out.

In [4]:
import inspect

from gambeta.cli import _rank_group as pipeline_stage

body = inspect.getsource(pipeline_stage)
print(f"does the ranking pipeline call shrink()?  {'shrink' in body}")
print(f"does it call zscore()?                    {'zscore' in body or '_normalise' in body}")

extreme = [
    describe("prior_minutes", v, rank_with(dataclasses.replace(cfg0, prior_minutes=v)))
    for v in (0.0, 10_000.0)
]
pd.DataFrame(extreme).style.format({"rho": "{:.4f}"}).hide(axis="index")

does the ranking pipeline call shrink()?  False
does it call zscore()?                    True


parameter,value,ranked,qualified,winner,top 10 kept,rho
prior_minutes,0.000000,5508,342,Lionel Messi,10/10,1.0000
prior_minutes,10000.000000,5508,342,Lionel Messi,10/10,1.0000


Setting the shrinkage prior to **zero** and to **ten thousand minutes** produces
byte-identical rankings, because `prior_minutes` is never read by the code that
builds them. `level.shrink` exists, it is tested, it is correct, and the only
thing that calls it is its own unit test.

So the honest position on that parameter is not "900 is well chosen". It is
**"there is no shrinkage in this pipeline"**, and the normalisation chapter
teaches it as though there were.

The gap is smaller than it sounds and it is not nothing. Careers are pooled with
each season weighted by its minutes, so a short season already counts for less
in the average. What is missing is the other half: a short season's *score* is
not pulled toward the mean, so a lucky 950-minute campaign enters at full
strength. That matters most for `consistency`, which takes the twentieth
percentile of a player's season scores without weighting them at all.

Two ways to close it, and they are not equivalent:

- **Wire the shrinkage in.** Faithful to what the book teaches, and it would
  move the answer. It also risks correcting twice, since minutes-weighted pooling
  is already a correction in the same direction.
- **Delete the parameter and the function**, and rewrite the normalisation
  chapter to describe minutes-weighted pooling, which is what actually happens.

I am not deciding that here, because the decision belongs in the open rather than
inside a chapter about sensitivity. What this chapter can say is that the
question was never asked until somebody swept the parameter and got a flat
line.